[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/01-python/04_loops.ipynb)

*Python from Scratch · Notebook 4 of 12*

# Loops (for / while)

Repeat work with for and while, control loops with break and continue, and nest loops for patterns.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A loop is a chore list repeated until done. Always ask: what changes each round, and what makes it stop?

**What interviewers really test here:** Off-by-one errors, loop invariants, break/continue, and the cost of nested loops.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Why loops? *(Basics)*
- [ ] 2. for loops and range() *(Basics)*
- [ ] 3. Accumulators and counters *(Core)*
- [ ] 4. while loops *(Core)*
- [ ] 5. break, continue and loop-else *(Core)*
- [ ] 6. Nested loops and patterns *(Advanced)*
- [ ] 7. Pythonic looping *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

A **loop** (iteration statement) repeatedly executes a block of code. Each execution of the block is one **iteration**. Loops are one of the three basic control structures, together with sequence and selection.




- A **definite loop** runs a known number of times or over a known collection — Python's `for`.
- An **indefinite loop** runs until a condition becomes false — Python's `while`.

### The for loop

```python
for variable in iterable:
    block
```



An **iterable** is any object that can give its items one at a time: str, list, tuple, dict (its keys), set, range, files, generators. On each iteration the next item is assigned to the loop variable.



Under the hood: Python calls `iter(iterable)` to get an **iterator**, then calls `next()` on it until it raises `StopIteration`.



#### range(start, stop, step)




- Produces integers from start (default 0) up to but **not including** stop, adding step (default 1).
- A negative step counts down: `range(10, 0, -2)` → 10 8 6 4 2.
- `range` is lazy — it doesn't build a list, so `range(10**9)` uses almost no memory.
- `len(range(a, b, s))` = ⌈(b − a) / s⌉ when positive.

### The while loop

```python
while condition:
    block
```



The condition is checked **before** every iteration (an **entry-controlled** loop). If it is False at the start, the block never runs. Python has no do-while; emulate it with `while True:` … `if not condition: break`.



A loop whose condition never becomes False is an **infinite loop**. Sometimes that's intentional (servers, games, `while True` with a `break`), usually it's a bug: the loop variable isn't updated.

### Loop control statements

| Statement | Effect |
| --- | --- |
| `break` | exits the innermost loop immediately |
| `continue` | skips the rest of the current iteration |
| `pass` | does nothing (placeholder) |
| `else` clause | runs after the loop ends normally — not after `break` |
| `return` | inside a function, leaves the loop and the function |



`break` only leaves one level of nested loops. To leave several, put the loops in a function and `return`, or use a flag.

### Common loop patterns

| Pattern | Shape |
| --- | --- |
| Accumulate | `total = 0; for x in xs: total += x` |
| Count | `count = 0; for x in xs: if cond: count += 1` |
| Find max/min | `best = xs[0]; for x in xs: if x > best: best = x` |
| Search | `for x in xs: if x == t: break` (+ else) |
| Filter / transform | `out = []; for x in xs: if cond: out.append(f(x))` |
| Digit processing | `while n: d = n % 10; n //= 10` |
| Two-variable update | `a, b = b, a + b` (Fibonacci, GCD) |
| Nested (pairs / grids) | `for i in …: for j in …:` |

### Loops and running time

The cost of a loop is (number of iterations) × (cost of the body). This is the start of **time complexity**:



| Loop shape | Iterations | Big-O |
| --- | --- | --- |
| one loop over n items | n | O(n) |
| two nested loops over n | n × n | O(n²) |
| loop that halves n each time | log₂ n | O(log n) |
| loop up to √n | √n | O(√n) |



Hidden loops count too: `x in list`, `list.index`, `sum`, slicing and string concatenation in a loop are all O(n) each.

---
# Part 2: Lessons, step by step

## Lesson 1: Why loops?

*Level: Basics*

Printing the numbers 1 to 5 by hand needs 5 lines. Printing 1 to 1,000 would need 1,000 lines. A **loop** repeats a block of code for you.



```python
for i in range(1, 6):
    print(i)
```



Each repetition is called an **iteration**. The variable `i` is the **loop variable**; it takes the next value each time round.



Python has two loops: `for` (go through a known sequence of values) and `while` (keep going as long as a condition is true).

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def first_loop():
    for i in range(1, 6):
        print("Round", i)
    print("Loop finished")


first_loop()

Round 1
Round 2
Round 3
Round 4
Round 5
Loop finished


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 2: for loops and range()

*Level: Basics*

`range` produces a sequence of whole numbers. The **stop value is never included**.



| Call | Gives |
| --- | --- |
| `range(5)` | 0 1 2 3 4 |
| `range(2, 6)` | 2 3 4 5 |
| `range(0, 10, 3)` | 0 3 6 9 |
| `range(5, 0, -1)` | 5 4 3 2 1 |



A `for` loop can also walk through any **sequence** directly — the characters of a string, the items of a list:



```python
for ch in "cat":
    print(ch)          # c, a, t
for fruit in ["apple", "kiwi"]:
    print(fruit)
```

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def range_demo():
    evens = []
    for n in range(0, 10, 2):
        evens.append(n)
    print(evens)
    for ch in "cat":
        print(ch.upper())


range_demo()

[0, 2, 4, 6, 8]
C
A
T


**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q3](#q3). Try one now, or carry on and do them in Part 3.

## Lesson 3: Accumulators and counters

*Level: Core*

Most loops build up an answer. Create the variable **before** the loop, update it **inside**:



```python
total = 0               # sum accumulator starts at 0
count = 0               # counter starts at 0
product = 1             # product accumulator starts at 1
best = items[0]         # running maximum starts at the first item
for x in items:
    total += x
    if x > 0:
        count += 1
    product *= x
    if x > best:
        best = x
```



**⚠️ Watch out:** Putting `total = 0` inside the loop resets it every round — a classic bug.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def accumulator_demo():
    marks = [72, 45, 90, 38]
    total = 0
    passed = 0
    best = marks[0]
    for m in marks:
        total += m
        if m >= 40:
            passed += 1
        if m > best:
            best = m
    print("total", total, "passed", passed, "best", best)


accumulator_demo()

total 245 passed 3 best 90


**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q10](#q10), [Q3](#q3), [Q4](#q4). Try one now, or carry on and do them in Part 3.

## Lesson 4: while loops

*Level: Core*

A `while` loop repeats **as long as its condition is True**. Use it when you don't know in advance how many rounds you need.



```python
n = 1234
digits = 0
while n > 0:
    n //= 10
    digits += 1
```



Something inside the loop must eventually make the condition False. If not, you get an **infinite loop** (stop it with Ctrl + C).



Rule of thumb: **for** when you're walking through items or a fixed count; **while** when you're waiting for something to happen.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def while_demo():
    n = 1234
    digits = 0
    while n > 0:
        n //= 10
        digits += 1
    print("digits:", digits)


while_demo()

digits: 4


**✏️ Your turn:** you now have what you need for [Q15](#q15), [Q14](#q14), [Q16](#q16). Try one now, or carry on and do them in Part 3.

## Lesson 5: break, continue and loop-else

*Level: Core*

- `break` — leave the loop immediately.
- `continue` — skip the rest of this round and start the next one.
- `else` after a loop — runs only if the loop finished **without** a `break`. Handy for "search, and report if not found".



```python
for n in numbers:
    if n < 0:
        continue          # ignore negatives
    if n == target:
        print("found")
        break             # stop searching
else:
    print("not found")    # only if no break happened
```

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def break_continue_demo():
    numbers = [4, -2, 7, 9, 3]
    target = 9
    for n in numbers:
        if n < 0:
            continue
        if n == target:
            print("found", n)
            break
        print("checked", n)
    else:
        print("not found")


break_continue_demo()

checked 4
checked 7
found 9


**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q11](#q11), [Q13](#q13), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 6: Nested loops and patterns

*Level: Advanced*

A loop inside a loop: the inner loop runs **completely** for every round of the outer loop. With outer size n and inner size m, the body runs n × m times.



```python
for row in range(1, 4):
    line = ""
    for col in range(row):
        line += "*"
    print(line)
# *
# **
# ***
```



Nested loops are used for grids, tables, comparing every pair, and printing patterns. Be careful with big inputs: two nested loops over 10,000 items is 100 million steps.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [7]:
def nested_demo():
    for row in range(1, 4):
        line = ""
        for col in range(1, 4):
            line += str(row * col) + " "
        print(line)


nested_demo()

1 2 3 
2 4 6 
3 6 9 


**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q19](#q19), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 7: Pythonic looping

*Level: Advanced*

Python has tools that remove index bookkeeping:



| Instead of | Write |
| --- | --- |
| `for i in range(len(xs)): x = xs[i]` | `for x in xs:` |
| a separate counter | `for i, x in enumerate(xs):` |
| indexing two lists | `for a, b in zip(xs, ys):` |
| looping backwards with indices | `for x in reversed(xs):` |
| loop + append | `[f(x) for x in xs]` (comprehension) |
| loop + total | `sum(...)`, `max(...)`, `any(...)`, `all(...)` |



**⚠️ Watch out:** Don't add or remove items from a list while looping over it — loop over a copy (`for x in xs[:]`) or build a new list.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [8]:
def pythonic_demo():
    names = ["Ana", "Ben", "Cy"]
    scores = [88, 92, 79]
    for i, name in enumerate(names, start=1):
        print(i, name)
    for name, score in zip(names, scores):
        print(name, score)
    print(sum(scores), max(scores), any(s > 90 for s in scores))


pythonic_demo()

1 Ana
2 Ben
3 Cy
Ana 88
Ben 92
Cy 79
259 92 True


**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q5](#q5), [Q18](#q18). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Multiplication Table

🟢 Warm-up · *for · range*

Return the list `[n×1, n×2, …, n×10]`.

**Examples**

- `n = 3` → `[3, 6, 9, 12, 15, 18, 21, 24, 27, 30]`
- `n = 1` → `[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]`

In [ ]:
def table(n):
    """
    Return the list [n×1, n×2, …, n×10].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(table, [
    ((3,), [3, 6, 9, 12, 15, 18, 21, 24, 27, 30]),
    ((1,), [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A while loop with a manual counter and append.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A for loop over `range(1, 11)` inside a list comprehension.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A while loop with a manual counter and append.

```python
def table(n):
    result = []
    i = 1
    while i <= 10:
        result.append(n * i)
        i = i + 1
    return result
```

**Cost:** O(10) time, O(10) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A for loop over `range(1, 11)` inside a list comprehension.

```python
def table(n):
    return [n * i for i in range(1, 11)]
```

**Cost:** O(10) time, O(10) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A while loop with a manual counter and append. That costs O(10) time and O(10) space."

"I can do better: A for loop over `range(1, 11)` inside a list comprehension. That's O(10) time and O(10) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/multiplication-table)

<a id="q2"></a>
### Q2. Factorial

🟢 Warm-up · *Accumulator*

Return n! = 1 × 2 × … × n (0! = 1).

**Examples**

- `n = 5` → `120`
- `n = 0` → `1`
- `n = 10` → `3628800`

In [ ]:
def factorial(n):
    """
    Return n! = 1 × 2 × … × n (0! = 1).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(factorial, [
    ((5,), 120),
    ((0,), 1),
    ((10,), 3628800),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A while loop counting down and multiplying.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A for loop over range, or `math.prod(range(1, n + 1))`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A while loop counting down and multiplying.

```python
def factorial(n):
    result = 1
    while n > 1:
        result = result * n
        n = n - 1
    return result
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A for loop over range, or `math.prod(range(1, n + 1))`.

```python
def factorial(n):
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A while loop counting down and multiplying. That costs O(n) time and O(1) space."

"I can do better: A for loop over range, or `math.prod(range(1, n + 1))`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/factorial)

<a id="q3"></a>
### Q3. Count Vowels

🟢 Warm-up · *Loop over a string*

Return how many vowels (a, e, i, o, u, either case) are in `text`.

**Examples**

- `text = 'Hello World'` → `3`
- `text = 'PYTHON'` → `1`
- `text = 'rhythm'` → `0`

In [ ]:
def count_vowels(text):
    """
    Return how many vowels (a, e, i, o, u, either case) are in text.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_vowels, [
    (('Hello World',), 3),
    (('PYTHON',), 1),
    (('rhythm',), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over indices, compare each character with every vowel.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Loop over characters directly and count with `sum` and `in`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over indices, compare each character with every vowel.

```python
def count_vowels(text):
    count = 0
    for i in range(len(text)):
        ch = text[i].lower()
        if ch == "a" or ch == "e" or ch == "i" or ch == "o" or ch == "u":
            count += 1
    return count
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Loop over characters directly and count with `sum` and `in`.

```python
def count_vowels(text):
    return sum(1 for ch in text.lower() if ch in "aeiou")
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over indices, compare each character with every vowel. That costs O(n) time and O(1) space."

"I can do better: Loop over characters directly and count with `sum` and `in`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/count-vowels)

<a id="q4"></a>
### Q4. Sum of Even Numbers

🟢 Warm-up · *range with step*

Return the sum of all even numbers from 1 to n.

**Examples**

- `n = 10` → `30`
- `n = 7` → `12`
- `n = 1` → `0`

In [ ]:
def sum_even(n):
    """
    Return the sum of all even numbers from 1 to n.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sum_even, [
    ((10,), 30),
    ((7,), 12),
    ((1,), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over every number and add it when it's even.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`range(2, n + 1, 2)` produces only even numbers.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over every number and add it when it's even.

```python
def sum_even(n):
    total = 0
    for i in range(1, n + 1):
        if i % 2 == 0:
            total += i
    return total
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `range(2, n + 1, 2)` produces only even numbers.

```python
def sum_even(n):
    return sum(range(2, n + 1, 2))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over every number and add it when it's even. That costs O(n) time and O(1) space."

"I can do better: `range(2, n + 1, 2)` produces only even numbers. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/sum-of-even-numbers)

<a id="q5"></a>
### Q5. Fibonacci Sequence

🟢 Warm-up · *Two running variables*

Return the first n Fibonacci numbers: 0, 1, 1, 2, 3, 5, …

**Examples**

- `n = 7` → `[0, 1, 1, 2, 3, 5, 8]`
- `n = 1` → `[0]`
- `n = 2` → `[0, 1]`

In [ ]:
def fibonacci(n):
    """
    Return the first n Fibonacci numbers: 0, 1, 1, 2, 3, 5, …
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fibonacci, [
    ((7,), [0, 1, 1, 2, 3, 5, 8]),
    ((1,), [0]),
    ((2,), [0, 1]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build the list by reading the last two elements each time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep just two variables and update them together: `a, b = b, a + b`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Build the list by reading the last two elements each time.

```python
def fibonacci(n):
    seq = [0, 1]
    while len(seq) < n:
        seq.append(seq[len(seq) - 1] + seq[len(seq) - 2])
    return seq[:n]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Keep just two variables and update them together: `a, b = b, a + b`.

```python
def fibonacci(n):
    seq = []
    a, b = 0, 1
    for _ in range(n):
        seq.append(a)
        a, b = b, a + b
    return seq
```

**Cost:** O(n) time, O(n) output space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build the list by reading the last two elements each time. That costs O(n) time and O(n) space."

"I can do better: Keep just two variables and update them together: `a, b = b, a + b`. That's O(n) time and O(n) output space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/fibonacci-sequence)

<a id="q6"></a>
### Q6. Countdown

🟢 Warm-up · *range backwards*

Return the list `[n, n−1, …, 1]`.

**Examples**

- `n = 5` → `[5, 4, 3, 2, 1]`
- `n = 1` → `[1]`
- `n = 0` → `[]`

In [ ]:
def countdown(n):
    """
    Return the list [n, n−1, …, 1].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(countdown, [
    ((5,), [5, 4, 3, 2, 1]),
    ((1,), [1]),
    ((0,), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A while loop that appends and decrements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`range(n, 0, -1)` counts down directly.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A while loop that appends and decrements.

```python
def countdown(n):
    out = []
    while n > 0:
        out.append(n)
        n -= 1
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `range(n, 0, -1)` counts down directly.

```python
def countdown(n):
    return list(range(n, 0, -1))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A while loop that appends and decrements. That costs O(n) time and O(n) space."

"I can do better: `range(n, 0, -1)` counts down directly. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/countdown)

<a id="q7"></a>
### Q7. First Position of a Value

🟢 Warm-up · *break · enumerate*

Return the index of the first `target` in `items`, or −1 if it isn't there.

**Examples**

- `items = [4, 9, 2, 9], target = 9` → `1`
- `items = [1, 2, 3], target = 7` → `-1`
- `items = ['a', 'b'], target = 'a'` → `0`

In [ ]:
def first_index(items, target):
    """
    Return the index of the first target in items, or −1 if it isn't there.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(first_index, [
    (([4, 9, 2, 9], 9), 1),
    (([1, 2, 3], 7), -1),
    ((['a', 'b'], 'a'), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over indices with a 'found' flag and break.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`enumerate` gives index and value; return as soon as you find it.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over indices with a 'found' flag and break.

```python
def first_index(items, target):
    found = -1
    i = 0
    while i < len(items):
        if items[i] == target:
            found = i
            break
        i += 1
    return found
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `enumerate` gives index and value; return as soon as you find it.

```python
def first_index(items, target):
    for i, x in enumerate(items):
        if x == target:
            return i
    return -1
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over indices with a 'found' flag and break. That costs O(n) time and O(1) space."

"I can do better: `enumerate` gives index and value; return as soon as you find it. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/first-position-of-a-value)

<a id="q8"></a>
### Q8. Star Triangle

🟢 Warm-up · *Nested loops · patterns*

Return the rows of a right triangle of stars of height n, e.g. n = 3 → ['\*', '\*\*', '\*\*\*'].

**Examples**

- `n = 3` → `['*', '**', '***']`
- `n = 1` → `['*']`

In [ ]:
def triangle_rows(n):
    """
    Return the rows of a right triangle of stars of height n, e.g. n = 3 → ['*',
    '**', '***'].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(triangle_rows, [
    ((3,), ['*', '**', '***']),
    ((1,), ['*']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An outer loop for rows and an inner loop that adds one star at a time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

String repetition `'*' * i` replaces the inner loop.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** An outer loop for rows and an inner loop that adds one star at a time.

```python
def triangle_rows(n):
    rows = []
    for i in range(1, n + 1):
        row = ""
        for j in range(i):
            row = row + "*"
        rows.append(row)
    return rows
```

**Cost:** O(n²) time, O(n²) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** String repetition `'*' * i` replaces the inner loop.

```python
def triangle_rows(n):
    return ["*" * i for i in range(1, n + 1)]
```

**Cost:** O(n²) time, O(n²) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An outer loop for rows and an inner loop that adds one star at a time. That costs O(n²) time and O(n²) space."

"I can do better: String repetition `'*' * i` replaces the inner loop. That's O(n²) time and O(n²) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/star-triangle)

<a id="q9"></a>
### Q9. Count Multiples in a Range

🟢 Warm-up · *Loop vs formula*

How many numbers in [a, b] are divisible by k? (a ≥ 1)

**Examples**

- `a = 1, b = 10, k = 3` → `3`
- `a = 5, b = 25, k = 5` → `5`
- `a = 7, b = 9, k = 10` → `0`

In [ ]:
def count_multiples(a, b, k):
    """
    How many numbers in [a, b] are divisible by k? (a ≥ 1)
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_multiples, [
    ((1, 10, 3), 3),
    ((5, 25, 5), 5),
    ((7, 9, 10), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check every number in the range.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Multiples up to b minus multiples below a: `b // k − (a − 1) // k`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check every number in the range.

```python
def count_multiples(a, b, k):
    count = 0
    for x in range(a, b + 1):
        if x % k == 0:
            count += 1
    return count
```

**Cost:** O(b − a) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Multiples up to b minus multiples below a: `b // k − (a − 1) // k`.

```python
def count_multiples(a, b, k):
    return b // k - (a - 1) // k
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check every number in the range. That costs O(b − a) time and O(1) space."

"I can do better: Multiples up to b minus multiples below a: `b // k − (a − 1) // k`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/count-multiples-in-a-range)

<a id="q10"></a>
### Q10. Largest in a List

🟢 Warm-up · *Running maximum*

Return the largest number in a non-empty list.

**Examples**

- `items = [3, 9, 2, 7]` → `9`
- `items = [-5, -2, -9]` → `-2`
- `items = [4]` → `4`

In [ ]:
def largest_in(items):
    """
    Return the largest number in a non-empty list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(largest_in, [
    (([3, 9, 2, 7],), 9),
    (([-5, -2, -9],), -2),
    (([4],), 4),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the largest so far while looping.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The built-in `max()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Track the largest so far while looping.

```python
def largest_in(items):
    best = items[0]
    for x in items[1:]:
        if x > best:
            best = x
    return best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** The built-in `max()`.

```python
def largest_in(items):
    return max(items)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the largest so far while looping. That costs O(n) time and O(1) space."

"I can do better: The built-in `max()`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/largest-in-a-list)

<a id="q11"></a>
### Q11. Sum Skipping Negatives

🟢 Warm-up · *continue*

Return the sum of the non-negative numbers in a list.

**Examples**

- `items = [3, -1, 4, -5, 2]` → `9`
- `items = [-1, -2]` → `0`
- `items = []` → `0`

In [ ]:
def sum_non_negative(items):
    """
    Return the sum of the non-negative numbers in a list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sum_non_negative, [
    (([3, -1, 4, -5, 2],), 9),
    (([-1, -2],), 0),
    (([],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Use `continue` to skip negatives inside the loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A generator with a filter inside `sum()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Use `continue` to skip negatives inside the loop.

```python
def sum_non_negative(items):
    total = 0
    for x in items:
        if x < 0:
            continue
        total += x
    return total
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A generator with a filter inside `sum()`.

```python
def sum_non_negative(items):
    return sum(x for x in items if x >= 0)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Use `continue` to skip negatives inside the loop. That costs O(n) time and O(1) space."

"I can do better: A generator with a filter inside `sum()`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/sum-skipping-negatives)

<a id="q12"></a>
### Q12. Smallest Divisor

🟢 Warm-up · *for … else*

Return the smallest divisor of n greater than 1 (n ≥ 2). A prime is its own smallest divisor.

**Examples**

- `n = 15` → `3`
- `n = 13` → `13`
- `n = 8` → `2`

In [ ]:
def smallest_divisor(n):
    """
    Return the smallest divisor of n greater than 1 (n ≥ 2). A prime is its own
    smallest divisor.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(smallest_divisor, [
    ((15,), 3),
    ((13,), 13),
    ((8,), 2),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Collect every divisor in a list, then take the first one.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Return at the first divisor found; it only needs to check up to √n.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Collect every divisor in a list, then take the first one.

```python
def smallest_divisor(n):
    divisors = []
    for d in range(2, n + 1):
        if n % d == 0:
            divisors.append(d)
    return divisors[0]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Return at the first divisor found; it only needs to check up to √n.

```python
def smallest_divisor(n):
    d = 2
    while d * d <= n:
        if n % d == 0:
            return d
        d += 1
    return n
```

**Cost:** O(√n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Collect every divisor in a list, then take the first one. That costs O(n) time and O(n) space."

"I can do better: Return at the first divisor found; it only needs to check up to √n. That's O(√n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/smallest-divisor)

<a id="q13"></a>
### Q13. Is It Prime?

🟡 Interview standard · *Loop with early exit*

Return True if n is a prime number (only divisible by 1 and itself).

**Examples**

- `n = 7` → `True`
- `n = 12` → `False`
- `n = 1` → `False`

In [ ]:
def is_prime(n):
    """
    Return True if n is a prime number (only divisible by 1 and itself).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_prime, [
    ((7,), True),
    ((12,), False),
    ((1,), False),
    ((97,), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every divisor from 2 to n − 1.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Divisors come in pairs, so testing up to √n is enough; stop at the first divisor found.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Try every divisor from 2 to n − 1.

```python
def is_prime(n):
    if n < 2:
        return False
    for d in range(2, n):
        if n % d == 0:
            return False
    return True
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Divisors come in pairs, so testing up to √n is enough; stop at the first divisor found.

```python
def is_prime(n):
    if n < 2:
        return False
    d = 2
    while d * d <= n:
        if n % d == 0:
            return False
        d += 1
    return True
```

**Cost:** O(√n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every divisor from 2 to n − 1. That costs O(n) time and O(1) space."

"I can do better: Divisors come in pairs, so testing up to √n is enough; stop at the first divisor found. That's O(√n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/is-it-prime)

<a id="q14"></a>
### Q14. Greatest Common Divisor

🟡 Interview standard · *while loop · Euclid*

Return the GCD of two positive integers.

**Examples**

- `a = 48, b = 18` → `6`
- `a = 17, b = 5` → `1`
- `a = 100, b = 75` → `25`

In [ ]:
def gcd(a, b):
    """
    Return the GCD of two positive integers.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(gcd, [
    ((48, 18), 6),
    ((17, 5), 1),
    ((100, 75), 25),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count down from the smaller number until one divides both.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Euclid's algorithm: replace (a, b) with (b, a % b) until b is 0.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Count down from the smaller number until one divides both.

```python
def gcd(a, b):
    d = min(a, b)
    while d > 1:
        if a % d == 0 and b % d == 0:
            return d
        d -= 1
    return 1
```

**Cost:** O(min(a, b)) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Euclid's algorithm: replace (a, b) with (b, a % b) until b is 0.

```python
def gcd(a, b):
    while b:
        a, b = b, a % b
    return a
```

**Cost:** O(log min(a, b)) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count down from the smaller number until one divides both. That costs O(min(a, b)) time and O(1) space."

"I can do better: Euclid's algorithm: replace (a, b) with (b, a % b) until b is 0. That's O(log min(a, b)) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/greatest-common-divisor)

<a id="q15"></a>
### Q15. Collatz Steps

🟡 Interview standard · *while with a changing condition*

Starting at n, if n is even halve it, otherwise make it 3n + 1. Return how many steps it takes to reach 1.

**Examples**

- `n = 6` → `8`
- `n = 1` → `0`
- `n = 27` → `111`

In [ ]:
def collatz_steps(n):
    """
    Starting at n, if n is even halve it, otherwise make it 3n + 1. Return how
    many steps it takes to reach 1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(collatz_steps, [
    ((6,), 8),
    ((1,), 0),
    ((27,), 111),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Store every value in a list, then the answer is its length − 1.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Only a counter is needed — no list.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Store every value in a list, then the answer is its length − 1.

```python
def collatz_steps(n):
    sequence = [n]
    while sequence[-1] != 1:
        last = sequence[-1]
        if last % 2 == 0:
            sequence.append(last // 2)
        else:
            sequence.append(3 * last + 1)
    return len(sequence) - 1
```

**Cost:** O(steps) time, O(steps) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Only a counter is needed — no list.

```python
def collatz_steps(n):
    steps = 0
    while n != 1:
        n = n // 2 if n % 2 == 0 else 3 * n + 1
        steps += 1
    return steps
```

**Cost:** O(steps) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Store every value in a list, then the answer is its length − 1. That costs O(steps) time and O(steps) space."

"I can do better: Only a counter is needed — no list. That's O(steps) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/collatz-steps)

<a id="q16"></a>
### Q16. Digital Root

🟡 Interview standard · *Nested while loops*

Keep adding the digits of n until a single digit remains (e.g. 9875 → 29 → 11 → 2). Return it.

**Examples**

- `n = 9875` → `2`
- `n = 5` → `5`
- `n = 0` → `0`

In [ ]:
def digital_root(n):
    """
    Keep adding the digits of n until a single digit remains (e.g. 9875 → 29 →
    11 → 2). Return it.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(digital_root, [
    ((9875,), 2),
    ((5,), 5),
    ((0,), 0),
    ((18,), 9),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeat the digit-sum loop while the number has more than one digit.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Maths shortcut: the digital root is 1 + (n − 1) % 9 for n &gt; 0.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Repeat the digit-sum loop while the number has more than one digit.

```python
def digital_root(n):
    while n >= 10:
        total = 0
        while n > 0:
            total += n % 10
            n //= 10
        n = total
    return n
```

**Cost:** O(d · rounds) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Maths shortcut: the digital root is 1 + (n − 1) % 9 for n &gt; 0.

```python
def digital_root(n):
    return 0 if n == 0 else 1 + (n - 1) % 9
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeat the digit-sum loop while the number has more than one digit. That costs O(d · rounds) time and O(1) space."

"I can do better: Maths shortcut: the digital root is 1 + (n − 1) % 9 for n &gt; 0. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/digital-root)

<a id="q17"></a>
### Q17. Armstrong Number

🟡 Interview standard · *Loop over digits*

An Armstrong number equals the sum of its digits each raised to the number of digits (153 = 1³ + 5³ + 3³). Return True or False.

**Examples**

- `n = 153` → `True`
- `n = 9474` → `True`
- `n = 123` → `False`

In [ ]:
def is_armstrong(n):
    """
    An Armstrong number equals the sum of its digits each raised to the number
    of digits (153 = 1³ + 5³ + 3³). Return True or False.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_armstrong, [
    ((153,), True),
    ((9474,), True),
    ((123,), False),
    ((5,), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count the digits in one loop, then sum the powers in a second loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use the string form: its length is the power, and a generator sums the powers.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Count the digits in one loop, then sum the powers in a second loop.

```python
def is_armstrong(n):
    count = 0
    temp = n
    while temp > 0:
        count += 1
        temp //= 10
    total = 0
    temp = n
    while temp > 0:
        total += (temp % 10) ** count
        temp //= 10
    return total == n
```

**Cost:** O(d) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Use the string form: its length is the power, and a generator sums the powers.

```python
def is_armstrong(n):
    digits = str(n)
    return n == sum(int(d) ** len(digits) for d in digits)
```

**Cost:** O(d) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count the digits in one loop, then sum the powers in a second loop. That costs O(d) time and O(1) space."

"I can do better: Use the string form: its length is the power, and a generator sums the powers. That's O(d) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/armstrong-number)

<a id="q18"></a>
### Q18. Perfect Number

🟡 Interview standard · *Divisor pairs*

A perfect number equals the sum of its divisors other than itself (6 = 1 + 2 + 3). Return True or False.

**Examples**

- `n = 6` → `True`
- `n = 28` → `True`
- `n = 12` → `False`

In [ ]:
def is_perfect(n):
    """
    A perfect number equals the sum of its divisors other than itself (6 = 1 + 2
    + 3). Return True or False.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_perfect, [
    ((6,), True),
    ((28,), True),
    ((12,), False),
    ((1,), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add every divisor from 1 to n − 1.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Divisors come in pairs (d, n // d), so loop only up to √n and add both.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Add every divisor from 1 to n − 1.

```python
def is_perfect(n):
    total = 0
    for d in range(1, n):
        if n % d == 0:
            total += d
    return total == n
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Divisors come in pairs (d, n // d), so loop only up to √n and add both.

```python
def is_perfect(n):
    if n < 2:
        return False
    total = 1
    d = 2
    while d * d <= n:
        if n % d == 0:
            total += d
            if d != n // d:
                total += n // d
        d += 1
    return total == n
```

**Cost:** O(√n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add every divisor from 1 to n − 1. That costs O(n) time and O(1) space."

"I can do better: Divisors come in pairs (d, n // d), so loop only up to √n and add both. That's O(√n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/perfect-number)

<a id="q19"></a>
### Q19. Pairs With a Given Sum

🟡 Interview standard · *Nested loops vs set*

Count pairs (i &lt; j) of positions whose values add up to `target`.

**Examples**

- `items = [1, 5, 7, -1, 5], target = 6` → `3`
- `items = [1, 1, 1], target = 2` → `3`
- `items = [1, 2], target = 9` → `0`

In [ ]:
def count_pairs(items, target):
    """
    Count pairs (i < j) of positions whose values add up to target.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_pairs, [
    (([1, 5, 7, -1, 5], 6), 3),
    (([1, 1, 1], 2), 3),
    (([1, 2], 9), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check every pair with two nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One loop: for each value, add how many earlier values equal target − value (kept in a dict).

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check every pair with two nested loops.

```python
def count_pairs(items, target):
    count = 0
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i] + items[j] == target:
                count += 1
    return count
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** One loop: for each value, add how many earlier values equal target − value (kept in a dict).

```python
def count_pairs(items, target):
    seen = {}
    count = 0
    for x in items:
        count += seen.get(target - x, 0)
        seen[x] = seen.get(x, 0) + 1
    return count
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check every pair with two nested loops. That costs O(n²) time and O(1) space."

"I can do better: One loop: for each value, add how many earlier values equal target − value (kept in a dict). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/pairs-with-a-given-sum)

<a id="q20"></a>
### Q20. Primes Up To n

🔴 Stretch · *Sieve of Eratosthenes*

Return all primes ≤ n in increasing order.

**Examples**

- `n = 20` → `[2, 3, 5, 7, 11, 13, 17, 19]`
- `n = 2` → `[2]`
- `n = 1` → `[]`

In [ ]:
def primes_up_to(n):
    """
    Return all primes ≤ n in increasing order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(primes_up_to, [
    ((20,), [2, 3, 5, 7, 11, 13, 17, 19]),
    ((2,), [2]),
    ((1,), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Test each number with trial division.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sieve: assume everything is prime, then cross out multiples of each prime starting at p².

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Test each number with trial division.

```python
def primes_up_to(n):
    primes = []
    for x in range(2, n + 1):
        is_p = True
        for d in range(2, int(x ** 0.5) + 1):
            if x % d == 0:
                is_p = False
                break
        if is_p:
            primes.append(x)
    return primes
```

**Cost:** O(n√n) time, O(1) extra space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Sieve: assume everything is prime, then cross out multiples of each prime starting at p².

```python
def primes_up_to(n):
    if n < 2:
        return []
    is_p = [True] * (n + 1)
    is_p[0] = is_p[1] = False
    for p in range(2, int(n ** 0.5) + 1):
        if is_p[p]:
            for m in range(p * p, n + 1, p):
                is_p[m] = False
    return [x for x in range(n + 1) if is_p[x]]
```

**Cost:** O(n log log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Test each number with trial division. That costs O(n√n) time and O(1) extra space."

"I can do better: Sieve: assume everything is prime, then cross out multiples of each prime starting at p². That's O(n log log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/loops/primes-up-to-n)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What is the difference between for and while loops?</b></summary>

`for` iterates over the items of an iterable — use it when the sequence or count is known. `while` repeats as long as a condition is true — use it when you can't know the number of iterations in advance.

</details>

<details>
<summary><b>2. What does the else clause on a loop do?</b></summary>

It runs once after the loop finishes normally (the iterable is exhausted or the while condition becomes false). It is skipped if the loop exits with `break`. It's useful for search loops: 'not found'.

</details>

<details>
<summary><b>3. break vs continue vs pass?</b></summary>

`break` exits the loop. `continue` jumps to the next iteration. `pass` does nothing and just fills a required spot in the syntax.

</details>

<details>
<summary><b>4. Is range(5) a list?</b></summary>

No. It's a lazy range object that produces numbers on demand and uses constant memory. Use `list(range(5))` if you really need a list.

</details>

<details>
<summary><b>5. Why is modifying a list while iterating over it a problem?</b></summary>

The iterator walks by index; removing items shifts later items left so some are skipped, and appending can make the loop run forever. Iterate over a copy or build a new list instead.

</details>

<details>
<summary><b>6. Does Python have a do-while loop?</b></summary>

No. Use `while True:` with the body first and `if not condition: break` at the end.

</details>

<details>
<summary><b>7. How do you loop with an index in Python?</b></summary>

Use `enumerate`: `for i, x in enumerate(items):`. Use `range(len(items))` only when you really need to modify by index.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Why loops?
- [ ] I can explain: for loops and range()
- [ ] I can explain: Accumulators and counters
- [ ] I can explain: while loops
- [ ] I can explain: break, continue and loop-else
- [ ] I can explain: Nested loops and patterns
- [ ] I can explain: Pythonic looping

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Digital Root
- [ ] [Q17](#q17) Armstrong Number
- [ ] [Q18](#q18) Perfect Number
- [ ] [Q19](#q19) Pairs With a Given Sum
- [ ] [Q20](#q20) Primes Up To n

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.